# 09 – Recommender System: ALS Collaborative Filtering

Implements a book recommender using PySpark MLlib's **ALS** (Alternating Least Squares):
- Collaborative filtering on explicit ratings (rating > 0)
- ISBN string indexing for ALS integer requirement
- 80/20 train-test split
- MLflow experiment tracking (parameters + metrics + model artifact)
- Top-N recommendations per user saved to gold layer

**Source:** `gold.book_ratings`
**Target:** `gold.recommendations`
**Config:** `config/09_recommender_model.json`

In [0]:
import json, sys, importlib

SETUP_CONFIG = "/Workspace/Users/tharcisio.leone@x1f-fink.one/poc/config/00_setup.json"
with open(SETUP_CONFIG) as f:
    setup = json.load(f)

sys.path.insert(0, setup["project_path"])

for mod in [m for m in list(sys.modules.keys()) if m.startswith("source")]:
    del sys.modules[mod]

RecommenderModel = importlib.import_module("source.09_recommender_model").RecommenderModel

CONFIG_PATH = f"{setup['project_path']}/config/09_recommender_model.json"

In [0]:
pipeline = RecommenderModel(config_path=CONFIG_PATH)
results = pipeline.run()

print(f"\n=== Model Metrics ===")
for metric, value in results["metrics"].items():
    print(f"  {metric}: {value}")

In [0]:
display(results["recommendations"])

In [0]:
from pyspark.sql import functions as F

sample_user = results["recommendations"].select("user_id").first()[0]
user_recs = results["recommendations"].filter(F.col("user_id") == sample_user)
print(f"\nTop recommendations for user {sample_user}:")
display(user_recs)